# 08 — Model Selection & Interpretation

**Phase 8: Model Selection & Recommendations** · Jassim M.N.M (IT24103137) · Group AI-39

This notebook selects the final model and explains which pre-match factors drive its predictions. All models use the shared chronological split (training: 2009/10–2014/15; test: 2015/16) and configurations selected on the 2014/15 validation season (D-021, D-028). The selected hyperparameters were fixed using the 2014/15 validation season. The 2015/16 test season provides the final model comparison; this notebook records the model choice before running feature-importance interpretation and performs no further tuning.


In [1]:
from pathlib import Path
import ast
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from sklearn.inspection import permutation_importance

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src.train_models import (
    FEATURE_COLUMNS, NUMERIC_FEATURE_COLUMNS, TEST_SEASON,
    chronological_split, fit_models, load_features,
)

print("scikit-learn version:", sklearn.__version__)
TABLES = REPO_ROOT / "outputs" / "tables"
FIGURES = REPO_ROOT / "outputs" / "figures"
features = load_features(REPO_ROOT / "data" / "processed" / "features.csv")
prepared = chronological_split(features, test_season=TEST_SEASON)

# Reuse configurations selected on 2014/15. Do not retune on the test season.
training_summary = pd.read_csv(TABLES / "model_training_summary.csv")
configs = {row.model: ast.literal_eval(row.selected_config) for row in training_summary.itertuples()}
trained = fit_models(prepared, configs=configs)

# Confirm the selected estimators reproduce the saved test predictions.
saved = pd.read_csv(REPO_ROOT / "outputs" / "predictions" / "test_predictions.csv")
for name in ["logistic_regression", "random_forest"]:
    match = (trained.predictions[f"{name}_prediction"].to_numpy() == saved[f"{name}_prediction"].to_numpy()).mean()
    print(f"{name}: {match:.1%} of saved test predictions identical")


scikit-learn version: 1.9.1
logistic_regression: 100.0% of saved test predictions identical
random_forest: 100.0% of saved test predictions identical


In [2]:
comparison = pd.read_csv(TABLES / "model_comparison.csv").set_index("model")
per_class = pd.read_csv(TABLES / "per_class_metrics.csv")
draw = per_class[per_class["class"] == "Draw"].set_index("model")[["recall", "f1"]].add_prefix("draw_")
selection = comparison[["accuracy", "macro_f1"]].join(draw).sort_values("macro_f1", ascending=False)
print(selection.round(4).to_string())


                     accuracy  macro_f1  draw_recall  draw_f1
model                                                        
logistic_regression    0.4546    0.4342       0.2807   0.2837
random_forest          0.4438    0.4341       0.3649   0.3203
decision_tree          0.4281    0.4265       0.4234   0.3330
knn                    0.4308    0.3893       0.1848   0.2175
majority_baseline      0.4387    0.2033       0.0000   0.0000


## Model selection

**Selected model: Logistic Regression (balanced, C=1.0).** Logistic Regression and Random Forest are effectively tied on test macro F1 (0.434166 vs 0.434126; difference ≈0.00004). We choose Logistic Regression because it has slightly higher accuracy (45.5% vs 44.4%) and its coefficient directions are easier to explain to coaching staff. The small macro-F1 difference is not meaningful evidence that one model is better.

Random Forest remains a reasonable alternative if recognising more draws is the priority: its Draw recall is 36.5%, compared with 28.1% for Logistic Regression. The final choice favours clearer explanation while preserving the top macro-F1 result. This selection is based on the 2014/15 validation configuration and reported 2015/16 results; no further tuning was done on the test season.


In [3]:
# Interpretation only: after fixing the model choice, measure each feature's
# contribution on the held-out test season using macro-F1. A single feature is
# shuffled at a time; correlated features can share importance.
importance_rows = []
for name in ["logistic_regression", "random_forest"]:
    result = permutation_importance(
        trained.estimators[name], prepared.X_test, prepared.y_test,
        scoring="f1_macro", n_repeats=10, random_state=42, n_jobs=1,
    )
    importance_rows.append(pd.DataFrame({
        "model": name, "feature": FEATURE_COLUMNS,
        "macro_f1_drop_mean": result.importances_mean,
        "macro_f1_drop_std": result.importances_std,
    }))
perm = pd.concat(importance_rows, ignore_index=True)
perm.to_csv(TABLES / "permutation_importance.csv", index=False)
print(perm.pivot(index="feature", columns="model", values="macro_f1_drop_mean")
      .sort_values("logistic_regression", ascending=False).round(4).to_string())


model                     logistic_regression  random_forest
feature                                                     
Away_Win_Rate                          0.0203         0.0211
Home_Win_Rate                          0.0177         0.0203
Away_Avg_Goals_Last_5                  0.0038        -0.0001
Abs_Form_Diff                          0.0020        -0.0025
Home_Avg_Goals_Last_5                 -0.0001         0.0011
league_id                             -0.0002         0.0020
Away_Avg_Conceded_Last_5              -0.0024         0.0013
Form_Diff                             -0.0030         0.0017
Away_Form_Last_5                      -0.0047         0.0033
Home_Avg_Conceded_Last_5              -0.0047        -0.0025
Home_Form_Last_5                      -0.0056         0.0001


In [4]:
order = (perm[perm["model"] == "logistic_regression"]
         .sort_values("macro_f1_drop_mean")["feature"].tolist())
colors = {"logistic_regression": "#2a78d6", "random_forest": "#eb6834"}
labels = {"logistic_regression": "Logistic Regression", "random_forest": "Random Forest"}
fig, ax = plt.subplots(figsize=(9, 6))
y = np.arange(len(order))
for i, name in enumerate(["logistic_regression", "random_forest"]):
    d = perm[perm["model"] == name].set_index("feature").loc[order]
    ax.barh(y + (i - 0.5) * 0.4, d["macro_f1_drop_mean"], height=0.38,
            xerr=d["macro_f1_drop_std"], color=colors[name], label=labels[name],
            error_kw={"elinewidth": 1, "ecolor": "#555555"})
ax.axvline(0, color="#555555", linewidth=1)
ax.set_yticks(y, order)
ax.set_xlabel("Drop in test macro F1 when the feature is shuffled")
ax.set_title("Permutation importance on the 2015/16 test season", pad=32)
ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.01), frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(FIGURES / "phase8_permutation_importance.png", dpi=150)
print("Saved:", FIGURES / "phase8_permutation_importance.png")
plt.close(fig)


Saved: C:\Users\Nusri\Desktop\BD\football-match-outcome-ml\outputs\figures\phase8_permutation_importance.png


In [5]:
lr = trained.estimators["logistic_regression"]
coef = pd.DataFrame(
    lr.named_steps["classifier"].coef_,
    index=lr.named_steps["classifier"].classes_,
    columns=lr.named_steps["preprocessor"].get_feature_names_out(),
).T.loc[NUMERIC_FEATURE_COLUMNS, ["Home Win", "Draw", "Away Win"]]
coef.to_csv(TABLES / "logistic_regression_coefficients.csv")
print(coef.round(3).sort_values("Home Win", ascending=False).to_string())


                          Home Win   Draw  Away Win
Home_Win_Rate                0.261 -0.059    -0.201
Home_Avg_Goals_Last_5        0.106 -0.029    -0.077
Away_Avg_Conceded_Last_5     0.074 -0.007    -0.068
Abs_Form_Diff                0.033 -0.041     0.009
Away_Form_Last_5             0.016  0.021    -0.036
Home_Form_Last_5            -0.002  0.022    -0.020
Form_Diff                   -0.012  0.001     0.012
Home_Avg_Conceded_Last_5    -0.083  0.017     0.066
Away_Avg_Goals_Last_5       -0.105 -0.016     0.121
Away_Win_Rate               -0.215 -0.018     0.234


In [6]:
rf = trained.estimators["random_forest"]
rf_imp = pd.Series(
    rf.named_steps["classifier"].feature_importances_,
    index=rf.named_steps["preprocessor"].get_feature_names_out(),
)
league_total = rf_imp[[c for c in rf_imp.index if c.startswith("league_id")]].sum()
rf_imp = rf_imp[NUMERIC_FEATURE_COLUMNS]
rf_imp["league_id (11 leagues combined)"] = league_total
rf_imp = rf_imp.sort_values(ascending=False).rename("impurity_importance")
rf_imp.to_csv(TABLES / "random_forest_feature_importance.csv")
print(rf_imp.round(4).to_string())


Home_Win_Rate                      0.2341
Away_Win_Rate                      0.2127
Form_Diff                          0.0929
Away_Avg_Goals_Last_5              0.0703
Home_Form_Last_5                   0.0693
Home_Avg_Goals_Last_5              0.0666
league_id (11 leagues combined)    0.0647
Away_Form_Last_5                   0.0576
Away_Avg_Conceded_Last_5           0.0461
Home_Avg_Conceded_Last_5           0.0445
Abs_Form_Diff                      0.0410


## What drives the predictions — in football terms

1. **Venue-specific history is the strongest measured signal.** The home team's past home-win rate and the away team's past away-win rate are the top two features in the permutation analysis for both models. Shuffling either one lowers test macro F1 by about 0.02.
2. **Recent form still matters as context, but adds little unique signal once venue win rates are available.** Form and win rates partly measure the same underlying team strength. This does not mean recent form is irrelevant; its measured extra contribution in this feature-by-feature test is small.
3. **Interpretation caveat:** correlated features can share or mask importance when only one is shuffled at a time. Small or negative permutation scores do not prove a feature has no value.
4. **Draws remain hard to identify.** The absolute difference in last-five form has a small negative Logistic Regression coefficient for Draw; the signal is weak, consistent with Draw recall being lower than for the other classes.

The model is a decision-support aid for pre-match preparation, not a guaranteed result. See `docs/eda_to_recommendation.md` for the stakeholder recommendations and limitations.
